<a href="https://colab.research.google.com/github/Praneeth-official/LLMs--Simplified/blob/main/Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***1.Input Words***
# ***2.Embedding***
# ***3.Encoder GRU***
# ***4.Hidden State***
# ***5.Decoder GRU***
# ***6.Linear layer***
# ***7.Highest Scoring word***
# ***8.Next word***
# ***9. Output sentence***


In [7]:
import torch
import torch.nn as nn

# **1. Encoder**

In [25]:
class Encoder(nn.Module):
  def __init__(self,
               vocab_size,
               embedding_size,
               hidden_size
               ):
    super().__init__()

    # Convert word IDs into vectors
    self.embedding = nn.Embedding(
        vocab_size,
        embedding_size
    )

    # Read the input sequence by GRU (Gated Recurrent Unit)
    self.gru = nn.GRU(
        embedding_size,
        hidden_size
    )

  def forward(self,
                input_sequence
                ):
      # word IDs -> word vectors
      embedded_words = self.embedding(input_sequence)

      # GRU reads the Sequence
      output, hidden_state = self.gru(embedded_words)

      # Return the final memory
      return hidden_state


# **2. Decoder**

In [26]:
class Decoder(nn.Module):
  def __init__(self,
               vocab_size,
               embedding_size,
               hidden_size
               ):
    super().__init__()

    # Convert Output word IDs into vectors
    self.embedding = nn.Embedding(
        vocab_size,
        embedding_size
    )

    # Generate the output sequence
    self.gru = nn.GRU(
        embedding_size,
        hidden_size
    )

    # Convert GRU Output into word scores
    self.output_layer = nn.Linear(
        hidden_size,
        vocab_size
    )

  def forward(self,
              current_word,
              hidden_state
              ):

    # Add sequence dimension
    current_word = current_word.unsqueeze(0)

    # Word ID -> Vector
    embedded_word = self.embedding(current_word)

    # Generate next hidden state
    output, hidden_state = self.gru(
        embedded_word,
        hidden_state
    )

    # Convert hidden state into scores
    word_scores = self.output_layer(
        output.squeeze(0)
    )

    return word_scores, hidden_state

# **3. Seq2Seq**

In [27]:
from torch._prims_common import DeviceLikeType
class Seq2Seq(nn.Module):
  def __init__(self,
               encoder,
               decoder,
               device
               ):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.device = device

  def forward(
      self,
      input_sequence,
      target_sequence=None,
      max_length = 10,
      teacher_forcing_ratio = 0.5
  ):
    #--------
    # STEP-1: Encoder understands the input
    #--------

    hidden_state = self.encoder(input_sequence)

    #-------
    #STEP-2: Start the Decoder
    #-------

    batch_size = input_sequence.shape[1]

    # Start token
    current_word =torch.zeros(
        batch_size,
        dtype=torch.long
    ).to(self.device)

    generated_words = []

    #--------
    # STEP 3: Generate words one by one
    #--------

    for step in range(max_length):

      # Ask decoder for the next word
      word_scores, hidden_state = self.decoder(
          current_word,
          hidden_state
      )

      # Select the word with the highest score
      predicted_word = word_scores.argmax(1)

      # Save the prediction
      generated_words.append(
          predicted_word.unsqueeze(0)
      )

      #--------
      # STEP-4: Teacher forcing
      #--------

      use_teacher = (
          target_sequence is not None
          and step < target_sequence.shape[0]
          and torch.rand(1).item() < teacher_forcing_ratio
      )

      if use_teacher:
        # Give the decoder the correct word
        current_word = target_sequence[step]
      else:

        # Give the decoder its own prediction
        current_word = predicted_word

    # Combine all generated words
    generated_words = torch.cat(generated_words,dim = 0)
    return generated_words


# **4. Device**

In [28]:
from torch.backends.mps import is_available
# NVIDIA GPU - CUDA
# Apple Silicon Mac - MPS
# Otherwise - CPU

if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print("Using Device: ", device)


Using Device:  cpu


# **5. Model Setting**

In [29]:
VOCAB_SIZE = 10
EMBEDDING_SIZE = 8
HIDDEN_SIZE = 16
SEQUENCE_LENGTH = 5
BATCH_SIZE =2

# **6. Create Encoder**

In [30]:
from IPython.core.builtin_trap import HideBuiltin
encoder = Encoder(
    vocab_size = VOCAB_SIZE,
    embedding_size= EMBEDDING_SIZE,
    hidden_size= HIDDEN_SIZE
    )

# **7. Create Decoder**

In [31]:
decoder = Decoder(
    vocab_size= VOCAB_SIZE,
    embedding_size= EMBEDDING_SIZE,
    hidden_size= HIDDEN_SIZE
)

# **8. Connect Encoder + Decoder**

In [32]:
model = Seq2Seq(
    encoder = encoder,
    decoder = decoder,
    device = device
).to(device)

# **9. Create Sample Input**

In [33]:
input_sequence = torch.randint(
    1,
    VOCAB_SIZE,
    (SEQUENCE_LENGTH, BATCH_SIZE)
).to(device)

# **10. Create Sample Target**

In [34]:
target_sequence = torch.randint(
    1,
    VOCAB_SIZE,
    (SEQUENCE_LENGTH, BATCH_SIZE)
).to(device)


# **11. Run the Model**



In [35]:
output_sequence = model(
    input_sequence,
    target_sequence,
    max_length= SEQUENCE_LENGTH,
    teacher_forcing_ratio=0.7
)

# **12. Display Results**

In [37]:
print("\n Input")
print(input_sequence.T)
print("\n Target")
print(target_sequence.T)
print("Model Prediction")
print(output_sequence.T)


 Input
tensor([[7, 3, 5, 4, 1],
        [1, 1, 1, 1, 4]])

 Target
tensor([[6, 7, 9, 8, 9],
        [3, 2, 1, 1, 8]])
Model Prediction
tensor([[4, 0, 0, 0, 0],
        [4, 4, 3, 3, 3]])
